# 🔍 Basic RAG Pipeline — LangGraph + Ollama + ChromaDB

---

## 🧠 Same RAG idea, orchestrated with LangGraph instead of a manual loop

This notebook is the LangGraph version of `basic_rag_langchain.ipynb`. The **indexing phase** (load → split → embed → store) is unchanged — LangGraph has nothing to add there, it's still plain document processing. What changes is the **querying phase**: instead of a hand-written `while True` loop that calls the retriever and then `ChatOllama` line by line, the retrieve-then-generate sequence becomes an explicit `StateGraph` with typed state, node functions, and edges — the same pattern used throughout this course from Day 3 onward.

**Two things are intentionally different from the LangChain notebook, to match this course's house style (see `Day-09-RAG-Engineering/example.py`):**
1. **`ollama.chat(...)` directly**, not `ChatOllama` — one fewer abstraction between you and the model call.
2. **A conditional edge**, not an `if` buried inside one function — when nothing relevant is retrieved, the graph *routes* to a separate `no_context` node instead of always paying for a model call on a miss.

```
┌─────────────────────────────────────────────────────────┐
│                  INDEXING PHASE (unchanged)              │
│  Raw Text File → Loader → Splitter → Embeddings → Chroma │
└─────────────────────────────────────────────────────────┘
              ↓  (query time)
┌─────────────────────────────────────────────────────────┐
│              QUERYING PHASE — now a StateGraph            │
│                                                           │
│        START                                              │
│          │                                                │
│          ▼                                                │
│      [ retrieve ]  ── looks up top-K chunks, builds state │
│          │                                                │
│          ▼ (conditional edge)                             │
│   context found? ──── no ───▶ [ no_context ] ──▶ END      │
│          │ yes                                            │
│          ▼                                                │
│      [ generate ]  ── ollama.chat(...) ──▶ END            │
└─────────────────────────────────────────────────────────┘
```

## 📦 Prerequisites & Installation

Same as the LangChain notebook, plus `langgraph`:

1. **Ollama installed and running** → https://ollama.com
2. **Models pulled:**
   ```bash
   ollama pull nomic-embed-text    # embedding model
   ollama pull gpt-oss:120b-cloud  # chat model
   ```
3. **Python packages:**
   ```bash
   pip install langchain-community langchain-text-splitters langchain-ollama langchain-chroma chromadb langgraph ollama
   ```
4. **A sample document at** `docs/company_policy.txt` (already present in this session folder).

---
## 📚 Step 0 — Imports

| Import | Role |
|---|---|
| `TextLoader` | Reads a `.txt` file from disk |
| `RecursiveCharacterTextSplitter` | Splits text into overlapping chunks |
| `OllamaEmbeddings` | Converts text → dense vectors |
| `Chroma` | Vector database — stores & searches embeddings |
| `ollama` | Direct client for the chat model — **no `ChatOllama` wrapper** |
| `StateGraph`, `START`, `END` | LangGraph's building blocks for the retrieve→generate pipeline |
| `TypedDict` | Defines the shape of the graph's state |

In [2]:
# ── Document Ingestion ───────────────────────────────────────────────────────────
from langchain_community.document_loaders.text import TextLoader

# ── Chunking ───────────────────────────────────────────────────────────────
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ── Embeddings (local, via Ollama) ──────────────────────────────────
from langchain_ollama import OllamaEmbeddings

# ── Vector Store (local, persisted to disk) ────────────────────────
from langchain_chroma import Chroma

# ── LLM — direct Ollama client, called from inside a graph node ──────────────
import ollama

# ── LangGraph ──────────────────────────────────────────────────────────────
from typing import TypedDict

from langgraph.graph import END, START, StateGraph

print("✅ All imports successful")

C:\Users\SR43993\AppData\Local\Temp\ipykernel_28544\1504769835.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.text import TextLoader


✅ All imports successful


---
## 📄 Step 1 — Load the Document

Identical to the LangChain notebook — `TextLoader` wraps the raw `.txt` file in a `Document` object carrying `page_content` and `metadata`.

In [3]:
print("Loading document...\n")

loader = TextLoader(
    "docs/company_policy.txt",
    encoding="utf-8"
)

documents = loader.load()

print(f"✅ Loaded {len(documents)} document(s)")
print(f"\n📄 Preview (first 300 chars):")
print(documents[0].page_content[:300])
print(f"\n🏷️  Metadata: {documents[0].metadata}")

Loading document...

✅ Loaded 1 document(s)

📄 Preview (first 300 chars):
Employees are eligible for 6 months maternity leave.

Employees must serve 60 days notice period before resignation.

Annual leave balance is 24 days.

Work from home is allowed twice a week.


🏷️  Metadata: {'source': 'docs/company_policy.txt'}


---
## ✂️ Step 2 — Split Document into Chunks

Same chunking logic and parameters as the LangChain notebook — chunking is an ingestion decision, not something LangGraph changes.

In [4]:
print("Splitting document into chunks...\n")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print(f"✅ Total Chunks Created: {len(chunks)}")
print(f"\n--- Previewing first 3 chunks ---")

for index, chunk in enumerate(chunks[:3]):
    print(f"\n🔹 Chunk {index + 1} ({len(chunk.page_content)} chars)")
    print(chunk.page_content)
    print("-" * 50)

Splitting document into chunks...

✅ Total Chunks Created: 1

--- Previewing first 3 chunks ---

🔹 Chunk 1 (191 chars)
Employees are eligible for 6 months maternity leave.

Employees must serve 60 days notice period before resignation.

Annual leave balance is 24 days.

Work from home is allowed twice a week.
--------------------------------------------------


---
## 🔢 Step 3 — Create the Embedding Model

`nomic-embed-text` via Ollama — unchanged from the LangChain notebook.

In [5]:
print("Loading Embedding Model...\n")

embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

print("✅ Embedding Model Ready")

test_vector = embeddings.embed_query("What is the leave policy?")
print(f"\n🔢 Vector dimension: {len(test_vector)}")
print(f"📊 First 5 values: {[round(v, 4) for v in test_vector[:5]]}")

Loading Embedding Model...

✅ Embedding Model Ready

🔢 Vector dimension: 768
📊 First 5 values: [0.0275, 0.037, -0.1446, -0.073, 0.0409]


---
## 🗄️ Step 4 — Create & Persist the Vector Store (ChromaDB)

Same as before. Stored in a separate folder (`./chroma_db_langgraph`) so this notebook doesn't collide with the LangChain notebook's `./chroma_db`.

In [6]:
print("Creating Chroma Vector Store...\n")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db_langgraph"
)

print("✅ Vector Store Created & Persisted to ./chroma_db_langgraph")
print(f"📦 Total vectors stored: {vectorstore._collection.count()}")

Creating Chroma Vector Store...

✅ Vector Store Created & Persisted to ./chroma_db_langgraph
📦 Total vectors stored: 1


---
## 🔎 Step 5 — Create the Retriever

Same `k=2` similarity retriever as the LangChain notebook.

In [7]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2}
)

print("✅ Retriever Ready")

test_query = "What is the leave policy?"
test_results = retriever.invoke(test_query)

print(f"\n🔍 Test Query: '{test_query}'")
print(f"📄 Retrieved {len(test_results)} chunk(s):")
for i, doc in enumerate(test_results):
    print(f"\n  Chunk {i+1}: {doc.page_content[:150]}...")

✅ Retriever Ready

🔍 Test Query: 'What is the leave policy?'
📄 Retrieved 1 chunk(s):

  Chunk 1: Employees are eligible for 6 months maternity leave.

Employees must serve 60 days notice period before resignation.

Annual leave balance is 24 days....


---
## 🧭 Step 6 — Define the Graph State

**This is where the LangGraph version diverges from the LangChain notebook.**

Instead of loose Python variables (`retrieved_docs`, `context`, `prompt`, `response`) passed by hand between lines of a loop, every value that flows through the pipeline is a field on one `State` object. Every node reads from `state` and returns only the keys it changes — the same contract used in every other day of this course.

| Field | Set by | Meaning |
|---|---|---|
| `question` | caller | the user's question |
| `context` | `retrieve` | joined text of the top-K retrieved chunks (empty string if nothing matched) |
| `answer` | `generate` or `no_context` | the final text shown to the user |

In [8]:
MODEL = "gpt-oss:120b-cloud"


class State(TypedDict):
    question: str
    context: str
    answer: str


print("✅ State schema defined")

✅ State schema defined


---
## 🧭 Step 7 — Node Functions

**`retrieve`** — calls the Chroma retriever (deterministic, no model call) and joins the returned chunks into one `context` string. If nothing comes back, `context` stays empty.

**`generate`** — builds the grounded prompt and calls `ollama.chat` directly (no `ChatOllama`), same guardrail prompt as the LangChain notebook: answer only from context, or say so.

**`no_context`** — new in this version. When `retrieve` finds nothing, the graph never calls the model at all — it routes straight to this node, which returns the fallback message for free. This is the kind of explicit stop condition LangGraph makes easy to see in the graph shape, versus buried in an `if` inside one big function.

In [9]:
def retrieve(state: State) -> dict:
    retrieved_docs = retriever.invoke(state["question"])
    context = "\n\n".join(doc.page_content for doc in retrieved_docs)
    return {"context": context}


def generate(state: State) -> dict:
    prompt = f"""You are a helpful assistant.

Answer ONLY using the provided context.
If the answer is not present in the context, reply exactly with:
"I could not find that information in the document."

Context:
{state['context']}

Question:
{state['question']}
"""
    response = ollama.chat(model=MODEL, messages=[{"role": "user", "content": prompt}])
    return {"answer": response["message"]["content"]}


def no_context(state: State) -> dict:
    return {"answer": "I could not find that information in the document."}


def route_after_retrieve(state: State) -> str:
    return "generate" if state["context"].strip() else "no_context"


print("✅ Node functions defined")

✅ Node functions defined


---
## 🔗 Step 8 — Build the Graph

```
START -> retrieve --(route_after_retrieve)--> generate -> END
                                          \--> no_context -> END
```

`add_conditional_edges` reads the return value of `route_after_retrieve` and follows the matching branch — this is the graph literally deciding, on every run, whether the model gets called at all.

In [10]:
graph_builder = StateGraph(State)

graph_builder.add_node("retrieve", retrieve)
graph_builder.add_node("generate", generate)
graph_builder.add_node("no_context", no_context)

graph_builder.add_edge(START, "retrieve")
graph_builder.add_conditional_edges(
    "retrieve",
    route_after_retrieve,
    {"generate": "generate", "no_context": "no_context"},
)
graph_builder.add_edge("generate", END)
graph_builder.add_edge("no_context", END)

graph = graph_builder.compile()

print("✅ Graph compiled")
print("Nodes:", list(graph.get_graph().nodes.keys()))

✅ Graph compiled
Nodes: ['__start__', 'retrieve', 'generate', 'no_context', '__end__']


---
## 💬 Step 9 — Single-Shot Query

One call to `graph.invoke(...)` now does what the LangChain notebook's manual retrieve → build-context → build-prompt → `llm.invoke` sequence did by hand.

In [11]:
question = "What is the work from home policy?"

result = graph.invoke({"question": question})

print(f"❓ Question: {question}")
print(f"\n📚 Retrieved Context:\n{result['context']}")
print(f"\n🤖 Answer: {result['answer']}")

❓ Question: What is the work from home policy?

📚 Retrieved Context:
Employees are eligible for 6 months maternity leave.

Employees must serve 60 days notice period before resignation.

Annual leave balance is 24 days.

Work from home is allowed twice a week.

🤖 Answer: Work from home is allowed twice a week.


---
## 🔁 Step 10 — Interactive Q&A Loop

Same interactive shell as the LangChain notebook, but each turn is one `graph.invoke` call instead of four hand-written steps. Try a question with no match in `docs/company_policy.txt` (e.g. "What is the firewall policy?") to see the `no_context` route fire without any model call.

In [12]:
print("RAG Q&A System Ready. Type 'exit' to stop.\n")
print("=" * 60)

while True:
    question = input("\n❓ Ask a question (or type 'exit'): ")

    if question.lower() == "exit":
        print("\n👋 Goodbye!")
        break

    result = graph.invoke({"question": question})

    print("\n===== 📚 RETRIEVED CONTEXT =====")
    print(result["context"] or "(nothing matched)")

    print("\n===== 🤖 ANSWER =====")
    print(result["answer"])

RAG Q&A System Ready. Type 'exit' to stop.


===== 📚 RETRIEVED CONTEXT =====
Employees are eligible for 6 months maternity leave.

Employees must serve 60 days notice period before resignation.

Annual leave balance is 24 days.

Work from home is allowed twice a week.

===== 🤖 ANSWER =====
Work from home is allowed twice a week.

===== 📚 RETRIEVED CONTEXT =====
Employees are eligible for 6 months maternity leave.

Employees must serve 60 days notice period before resignation.

Annual leave balance is 24 days.

Work from home is allowed twice a week.

===== 🤖 ANSWER =====
I could not find that information in the document.

===== 📚 RETRIEVED CONTEXT =====
Employees are eligible for 6 months maternity leave.

Employees must serve 60 days notice period before resignation.

Annual leave balance is 24 days.

Work from home is allowed twice a week.

===== 🤖 ANSWER =====
I could not find that information in the document.

👋 Goodbye!


---
## 🧪 Bonus — Load Existing Vector Store (Skip Re-indexing)

Once ChromaDB is persisted to disk, reload it and point the same `retriever` variable at it — the graph's `retrieve` node picks up the change automatically since it reads the global `retriever`.

In [13]:
vectorstore_loaded = Chroma(
    persist_directory="./chroma_db_langgraph",
    embedding_function=embeddings
)

retriever = vectorstore_loaded.as_retriever(search_kwargs={"k": 2})

print(f"✅ Loaded existing store from disk")
print(f"📦 Vectors in store: {vectorstore_loaded._collection.count()}")

result = graph.invoke({"question": "How many days notice period is required?"})
print(f"\n🤖 Answer: {result['answer']}")

✅ Loaded existing store from disk
📦 Vectors in store: 1

🤖 Answer: 60 days.


---
## 🎯 Key Takeaways — LangChain loop vs. LangGraph graph

| Concept | LangChain notebook | This notebook |
|---|---|---|
| Orchestration | Manual `while` loop, imperative steps | `StateGraph` with named nodes and edges |
| State | Loose local variables | One typed `State` dict passed between nodes |
| Model call | `ChatOllama(...).invoke(prompt)` | `ollama.chat(model=..., messages=[...])` directly |
| No-match handling | Always calls the model, prompt tells it to say "not found" | Conditional edge skips the model call entirely on a miss |
| Inspectability | Have to read the loop body to know the order of steps | The edge list *is* the order of steps |

Indexing (loader → splitter → embeddings → Chroma) is identical in both notebooks — LangGraph is a control-flow tool for the querying phase, not a replacement for ingestion.

## 🚀 Next Steps to Explore

1. **Memory** — add a checkpointer and `thread_id` (Day 7's pattern) so follow-up questions can refer back to earlier turns.
2. **Human-in-the-loop** — use `interrupt` (Day 8's pattern) to pause before `generate` and let a person approve or edit the retrieved context.
3. **Reranking node** — insert a node between `retrieve` and the routing step that re-scores the retrieved chunks with a cross-encoder before deciding whether to generate.
4. **Multi-source retrieval** — add a second retriever node and a router (Day 13's pattern) that picks which knowledge base to query based on the question.